<a href="https://colab.research.google.com/github/IBM/vLLM-Hook/blob/main/notebooks/demo_spotlight_long_conversation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight Long-Conversation Colab Smoke

This notebook validates Spotlight inference over a multi-turn Colab conversation using the CUDA/vLLM `HookLLM` path. It intentionally does not use the Apple Silicon/Metal backend.

### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.

In [1]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
import importlib
import importlib.metadata as importlib_metadata
import os
import re
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/IBM/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("VLLM_HOOK_REPO_DIR", "/content/vLLM-Hook"))
PLUGIN_SRC = REPO_DIR / "vllm_hook_plugins"

# These must be set before vLLM is imported for the first time in this kernel.
os.environ["VLLM_USE_V1"] = "1"
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "fork")
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")


def run(cmd, cwd=None, check=True):
    print("+ " + " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout, end="")
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout)
    return result


def normalized_package_name(requirement_line):
    line = requirement_line.split("#", 1)[0].strip()
    if not line or line.startswith("-"):
        return ""
    return re.split(r"[<>=!~;\[]", line, maxsplit=1)[0].strip().lower().replace("_", "-")


def version_tuple(version):
    parts = []
    for part in re.split(r"[.+-]", version):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def protobuf_needs_pin():
    try:
        current = importlib_metadata.version("protobuf")
    except importlib_metadata.PackageNotFoundError:
        return True
    parsed = version_tuple(current)
    return not ((5, 29, 6) <= parsed < (6, 30))


def fail_if_already_imported(package_names):
    imported = sorted(
        name for name in package_names
        if name in sys.modules or any(mod.startswith(name + ".") for mod in sys.modules)
    )
    if imported:
        raise RuntimeError(
            "The setup cell needs to run before importing "
            + ", ".join(imported)
            + ". Restart the runtime once, then use Runtime > Run all."
        )


if IN_COLAB:
    fail_if_already_imported(["torch", "torchvision", "torchaudio", "vllm"])

    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
    else:
        run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH])

    filtered_req = Path("/tmp/vllm_hook_colab_requirements.txt")
    req = REPO_DIR / "requirement.txt"
    if req.exists():
        keep = []
        skip_packages = {"vllm", "torch", "torchvision", "torchaudio", "protobuf", "pillow", "pil"}
        for line in req.read_text(encoding="utf-8").splitlines():
            if normalized_package_name(line) in skip_packages:
                continue
            keep.append(line)
        filtered_req.write_text("\n".join(keep) + "\n", encoding="utf-8")
        run([sys.executable, "-m", "pip", "install", "-r", filtered_req])

    if protobuf_needs_pin():
        if "google.protobuf" in sys.modules:
            raise RuntimeError(
                "Colab has already imported google.protobuf, but its installed protobuf "
                "version is outside the range needed by this notebook. Restart the runtime "
                "once, then run this setup cell before any other imports."
            )
        run([sys.executable, "-m", "pip", "install", "--upgrade", "protobuf>=5.29.6,<6.30"])

    run([sys.executable, "-m", "pip", "install", "-U", "uv"])

    # Current Colab GPU runtimes ship CUDA 12.8-era PyTorch wheels. Installing
    # torch and vLLM in one transaction keeps their ABI pins aligned and avoids
    # the old CUDA 13 resolver path without requiring a runtime restart.
    vllm_install_base = [
        sys.executable, "-m", "uv", "pip", "install",
        "--system", "--reinstall", "--no-cache", "--torch-backend=cu128",
        "torch", "torchvision", "torchaudio",
    ]
    exact_vllm = run(vllm_install_base + ["vllm==0.19.0"], check=False)
    if exact_vllm.returncode:
        print("vLLM 0.19.0 was not installable for this Colab runtime; falling back to the latest supported 0.18.x wheel.")
        run(vllm_install_base + ["vllm>=0.14,<0.19"])

    # Torchvision imports Pillow during vLLM/Transformers initialization. Colab
    # images can end up with mixed PIL files after large dependency changes in a
    # live kernel, so force a coherent Pillow install before validation imports.
    run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "pillow>=10.0"])
    for module_name in list(sys.modules):
        if module_name == "PIL" or module_name.startswith("PIL."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", PLUGIN_SRC])

    # Editable installs write .pth metadata that is normally consumed at interpreter
    # startup. Make the source tree importable immediately in this live kernel.
    plugin_path = str(PLUGIN_SRC)
    if plugin_path not in sys.path:
        sys.path.insert(0, plugin_path)
    importlib.invalidate_caches()
    os.chdir(REPO_DIR / "notebooks")

    import torch
    import vllm
    import vllm_hook_plugins

    print(f"torch {torch.__version__} (CUDA runtime: {torch.version.cuda})")
    print(f"vLLM {vllm.__version__}")
    print(f"vllm_hook_plugins loaded from {Path(vllm_hook_plugins.__file__).parent}")
    print("Setup complete. Continue with the next cell; no runtime restart is needed.")
else:
    print("Not running in Colab; install dependencies from the repository README if needed.")


+ git -C /content/vLLM-Hook fetch origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
+ git -C /content/vLLM-Hook checkout main
Already on 'main'
Your branch is up to date with 'origin/main'.
+ git -C /content/vLLM-Hook pull --ff-only origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
Already up to date.
+ /usr/bin/python3 -m pip install -r /tmp/vllm_hook_colab_requirements.txt
+ /usr/bin/python3 -m pip install --upgrade protobuf>=5.29.6,<6.30
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.1
    Uninstalling protobuf-7.36.1:
      Successfully uninstalled protobuf-7.36.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.75.3 requires pro

### Imports & Environment

In [ ]:
import io
import os
import multiprocessing as mp
import sys
import time
from pathlib import Path

import torch
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


### Initialize `HookLLM`

In [ ]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=0.65,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")
print(f"Max model length: {MAX_MODEL_LEN}")


### Long-Conversation Configuration

In [ ]:
PERSISTENT_INSTRUCTIONS = """Persistent instructions:
1. Answer in valid JSON with keys `turn_summary`, `answer`, and `constraint_check`.
2. Keep the `answer` concise, specific, and grounded in the recent conversation.
3. Mention the current topic exactly once in `turn_summary`.
""".strip()

SYSTEM_MESSAGE = (
    "You are continuing a long Talk2AI-style conversation. Preserve the persistent "
    "instructions even as the prompt accumulates conversation history."
)

SEED_HISTORY = [
    {"role": "user", "content": "I keep hearing that climate change discussions are exaggerated compared with other problems."},
    {"role": "assistant", "content": "Let's compare the long-run measurements with the way headlines describe them."},
    {"role": "user", "content": "The headlines make me distrust the whole topic."},
    {"role": "assistant", "content": "That reaction makes sense; repeated alarm can make even strong evidence feel performative."},
]

USER_TURNS = [
    "What evidence should I look at if I want to avoid headline-driven conclusions?",
    "How do local weather experiences confuse the broader trend?",
    "What tradeoffs matter for households when policies raise energy costs?",
    "Where does adaptation make sense, and where does it fall short?",
    "How should poorer countries think about growth and emissions limits?",
    "What practical local policy question should I ask a city council candidate?",
    "How can I separate serious policy criticism from misinformation?",
    "End with a cautious, practical position that still respects the evidence.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=160)
MAX_USER_TURNS = min(8, len(USER_TURNS))
HISTORY_WINDOW_MESSAGES = 16


### Prompt And Run Helpers

In [ ]:
def render_long_conversation_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "
".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript += "
"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            "from this prompt to keep the Colab run within memory limits. Continue the same conversation.

"
        )

    return f"""{SYSTEM_MESSAGE}

{PERSISTENT_INSTRUCTIONS}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()


def clean_text(text):
    return (text or "").strip().split("

")[0].strip()


def run_long_conversation_condition(name, use_spotlight):
    history = list(SEED_HISTORY)
    rows = []
    for turn_index, user_message in enumerate(USER_TURNS[:MAX_USER_TURNS], start=1):
        prompt = render_long_conversation_prompt(history, user_message)
        started = time.perf_counter()
        if use_spotlight:
            outputs = generate_with_spotlight(
                llm,
                prompts=[prompt],
                emph_strings=[PERSISTENT_INSTRUCTIONS],
                alpha=ALPHA,
                sampling_params=SAMPLING_PARAMS,
            )
        else:
            outputs = llm.generate(
                prompts=[prompt],
                sampling_params=SAMPLING_PARAMS,
                use_hook=False,
            )
        latency_s = time.perf_counter() - started
        reply = clean_text(outputs[0].outputs[0].text)
        rows.append(
            {
                "condition": name,
                "turn": turn_index,
                "prompt_chars": len(prompt),
                "history_messages_in_prompt": min(len(history), HISTORY_WINDOW_MESSAGES),
                "latency_s": round(latency_s, 3),
                "user_message": user_message,
                "reply": reply,
            }
        )
        history.extend([
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": reply},
        ])
        print(f"{name} turn {turn_index}: prompt_chars={len(prompt)} latency={latency_s:.2f}s")
    return rows


### Run Baseline And Spotlight Conditions

In [ ]:
baseline_rows = run_long_conversation_condition("baseline", use_spotlight=False)
spotlight_rows = run_long_conversation_condition("spotlight", use_spotlight=True)
long_conversation_results = baseline_rows + spotlight_rows
long_conversation_results


### Smoke Summary And Limitations

In [ ]:
for condition in ["baseline", "spotlight"]:
    rows = [row for row in long_conversation_results if row["condition"] == condition]
    print("=" * 70)
    print(condition.upper())
    print("turns:", len(rows))
    print("max_prompt_chars:", max(row["prompt_chars"] for row in rows))
    print("mean_latency_s:", round(sum(row["latency_s"] for row in rows) / len(rows), 3))
    print("final_reply:")
    print(rows[-1]["reply"])

print("
Validation note: this notebook is intended for a Colab GPU runtime. Local smoke validation checks JSON structure and platform-specific imports only.")
